# Spike Global Epistasis Analysis

Visualize the sigmoidal global epistasis function and
prediction correlations for the chosen lasso strength.

In [ ]:
config_path = "config/config.yaml"
profile = None
run_name = None

In [ ]:
import warnings
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import multidms
import multidms.plot

from notebooks._common import load_config

In [ ]:
cfg = load_config(config_path, profile, run_name=run_name)
spike = cfg["spike"]
output_dir = spike["output_dir"]
chosen_lasso_strength = spike["chosen_lasso_strength"]

warnings.simplefilter("ignore")
%matplotlib inline
plt.rcParams.update({"legend.frameon": False, "font.size": 11})

In [ ]:
models = pickle.load(open(f"{output_dir}/full_models.pkl", "rb"))
model_collection = multidms.ModelCollection(models)
print(f"Loaded {len(models)} models")

## Global epistasis landscape

In [ ]:
multidms.plot.ge_landscape(
    model_collection.fit_models
    .query(f"fusionreg == {chosen_lasso_strength}")
    .iloc[0]
    .model
)

## Prediction correlations

In [ ]:
chosen_replicate_models = models.query("fusionreg == @chosen_lasso_strength")
replicate_data = {}
for idx, row in chosen_replicate_models.iterrows():
    model = row.model
    replicate_data[row.dataset_name] = model.get_variants_df(phenotype_as_effect=False)

saveas = "global_epistasis_and_prediction_correlations"
fig, ax = plt.subplots(2, 2, figsize=[6.4, 6], sharex="col", sharey="col")

for i, (name, vdf) in enumerate(replicate_data.items()):
    # Latent vs func_score
    iter_ax = ax[i, 0]
    iter_ax.scatter(vdf["predicted_latent"], vdf["func_score"], alpha=0.05, s=1, c="grey")
    corr = pearsonr(vdf["predicted_latent"], vdf["func_score"])[0]
    iter_ax.annotate(f"$r = {corr:.2f}$", (0.05, 0.9), xycoords="axes fraction", fontsize=11)
    iter_ax.set_ylabel(f"{name}\nfunctional score")
    if i == 1:
        iter_ax.set_xlabel("predicted latent phenotype")

    # Predicted vs func_score
    iter_ax = ax[i, 1]
    iter_ax.scatter(vdf["predicted_func_score"], vdf["func_score"], alpha=0.05, s=1, c="grey")
    lims = [min(vdf["func_score"].min(), vdf["predicted_func_score"].min()),
            max(vdf["func_score"].max(), vdf["predicted_func_score"].max())]
    iter_ax.plot(lims, lims, "--", c="royalblue", lw=1)
    corr = pearsonr(vdf["predicted_func_score"], vdf["func_score"])[0]
    iter_ax.annotate(f"$r = {corr:.2f}$", (0.05, 0.9), xycoords="axes fraction", fontsize=11)
    if i == 1:
        iter_ax.set_xlabel("predicted functional score")

sns.despine()
plt.tight_layout()
fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()